<a href="https://colab.research.google.com/github/deguc/Shannon/blob/main/406_MC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%%
import numpy as np


class GridWorld:

    def __init__(self):

        self.H,self.W = 5,7
        self.action_size = 4
        self.grid = np.zeros((self.H,self.W),dtype=np.int8)
        self.grid[1,1:5] = 1
        self.grid[3,2:6] = 1
        self.grid[4,6] = 2
        self.agent_pos = np.array([0,0])
        self.memory = []

    def render(self):

        legend = np.array(['.','#','G'],dtype='>U1')
        view = legend[self.grid]

        for m in self.memory:
            view[tuple(m)] = '*'

        view[tuple(self.agent_pos)] = 'A'

        for v in view:
            print(' '.join(v))

        print()

    def onestep(self,state,action):

        move = np.array([[-1,0],[1,0],[0,-1],[0,1]])
        next_state = state + move[action]

        if not(0 <= next_state[0] < self.H and 0 <= next_state[1] <self.W):
            return state,-1.0,False

        cell = self.grid[tuple(next_state)]

        if cell == 1:
            return state,-1.0,False

        if cell == 2:
            return next_state,10,True

        return next_state,-0.1,False

    def step(self,action):

        self.memory += [self.agent_pos]
        next_state,reward,done = self.onestep(self.agent_pos,action)
        self.agent_pos = next_state

        return next_state,reward,done

    def reset(self,state=None):

        if state is None:
            state = np.array([0,0])

        self.agent_pos = state
        self.memory.clear()

        return state

class MCAgent:

    def __init__(self,H,W,action_size):

        self.H,self.W = H,W
        self.action_size = action_size
        self.gamma = 0.9
        self.alpha = 0.1

        self.Q = np.zeros((H,W,action_size),dtype=np.float32)

    def softmax(self,x):

        c = np.max(x,axis=-1,keepdims=True)
        z = np.exp(x-c)

        return z / np.sum(z,axis=-1,keepdims=True)

    def get_action(self,state,greedy=False):

        qs = self.Q[tuple(state)]

        if greedy:
            return np.argmax(qs)

        prob = self.softmax(qs)

        return np.random.choice(self.action_size,p=prob)

    def update(self,traj):

        G = 0
        for state,action,reward in reversed(traj):

            G = reward + self.gamma * G
            s = tuple(state)
            self.Q[s][action] += (G-self.Q[s][action])*self.alpha


env = GridWorld()
agent = MCAgent(env.H,env.W,env.action_size)

for _ in range(200):

    state = env.reset()
    traj = []

    for _ in range(200):

        action = agent.get_action(state)
        next_state,reward,done = env.step(action)
        traj.append((state,action,reward))

        if done:
            agent.update(traj)
            break

        state = next_state

state = env.reset()

for _ in range(20):

    action = agent.get_action(state,greedy=True)
    next_state,reward,done = env.step(action)

    if done:
        break

    state = next_state

env.render()
